In [0]:
%pip install databricks-labs-dqx
%pip install databricks-labs-dqx[llm]
%restart_python

In [0]:
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from databricks.sdk import WorkspaceClient
from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.rule import DQRule
import databricks.labs.dqx.llm as llm

In [0]:
env = 'dev'
SOURCE_TABLE     = f"bronze_{env}.salesforce_datatune.opportunity"
SILVER_TABLE     = f"silver_{env}.sales_ops.opportunity_dqx_test"
QUARANTINE_TABLE = f"silver_{env}.sales_ops.opportunity_dqx_quarantine"
MERGE_KEY        = "Id"   # Salesforce primary key

spark = SparkSession.getActiveSession()
ws    = WorkspaceClient()
dqe   = DQEngine(ws)

In [0]:
df = spark.table(SOURCE_TABLE)

In [0]:
# Auto-create warehouse and schema if missing
def ensure_schema_exists(catalog: str, schema: str):
    # PySpark SQL can execute DDL
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

# Extract from target table strings
def parse_table_fqn(table_fqn):
    parts = table_fqn.split('.')
    if len(parts) == 3:
        return parts[0], parts[1] # catalog, schema
    raise ValueError(f"Table name must be FQN: 'catalog.schema.table', got: {table_fqn}")

for target in [SILVER_TABLE, QUARANTINE_TABLE]:
    catalog, schema = parse_table_fqn(target)
    ensure_schema_exists(catalog, schema)

print("[INFO] Schemas checked/created as needed.")


In [0]:
from databricks.labs.dqx.rule import DQRowRule

def chk_amount_valid():
    """Amount must be non-null and strictly greater than zero."""
    c = F.col("Amount")
    return c.isNotNull() & (c > 0)

def chk_created_and_closed():
    """CreatedDate must be NOT NULL.
    If CloseDate is present it cannot be before CreatedDate.
    Passes automatically when CloseDate is null."""
    c_created = F.col("CreatedDate")
    c_closed = F.col("CloseDate")
    return (
        c_created.isNotNull() & (
            c_closed.isNull() | (c_closed >= c_created)
        )
    )

def chk_close_date_not_future_when_closed():
    """CloseDate must not be greater than today when IsClosed = true.
    Passes automatically when IsClosed is false/null."""
    return F.when(
        F.col("IsClosed").isNotNull() & (F.col("IsClosed") == True),
        F.col("CloseDate") <= F.current_date(),
    ).otherwise(F.lit(True))

checks = [
    DQRowRule(
        check_func=chk_amount_valid,
        name="amount_not_null_zero_or_negative",
        criticality="error"
    ),
    DQRowRule(
        check_func=chk_created_and_closed,
        name="created_and_closed_date_logic",
        criticality="error"
    ),
    DQRowRule(
        check_func=chk_close_date_not_future_when_closed,
        name="close_date_not_in_future_when_closed",
        criticality="error"
    ),
]


In [0]:
valid_df, quarantine_df = dqe.apply_checks_and_split(df, checks) 
(
    quarantine_df
    .withColumn("_dqx_run_ts", F.current_timestamp())
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)
print(f"[INFO] Quarantine write complete: {QUARANTINE_TABLE}")
 
silver_df = valid_df.withColumn("_dqx_loaded_ts", F.current_timestamp())

if not spark.catalog.tableExists(SILVER_TABLE):
    # First run: create the table from the valid dataset
    (
        silver_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(SILVER_TABLE)
    )
    print(f"[INFO] Silver table created on first run: {SILVER_TABLE}")
else:
    # Subsequent runs: MERGE on primary key — update existing, insert new
    target = DeltaTable.forName(spark, SILVER_TABLE)
    (
        target.alias("tgt")
        .merge(
            silver_df.alias("src"),
            f"tgt.{MERGE_KEY} = src.{MERGE_KEY}",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"[INFO] Incremental merge complete: {SILVER_TABLE}")




In [0]:
# Run summary 
valid_count      = valid_df.count()
quarantine_count = quarantine_df.count()
total_count      = valid_count + quarantine_count
quality_rate     = (valid_count / total_count * 100) if total_count > 0 else 0.0

print(f"\n{'─' * 55}")
print(f"  Source rows:      {total_count:>10,}")
print(f"  Silver rows:      {valid_count:>10,}")
print(f"  Quarantine rows:  {quarantine_count:>10,}")
print(f"  Quality rate:     {quality_rate:>9.1f}%")
print(f"{'─' * 55}\n")

In [0]:
# Determine which rule(s) each quarantined row failed
from pyspark.sql import functions as F

quarantine_with_flags = quarantine_df.withColumns({
    "violates_amount": ~chk_amount_valid()
})

totals = (quarantine_with_flags
    .select(F.col("violates_amount"))
    .agg(
        F.sum(F.col("violates_amount").cast("int")).alias("failed_amount") 
    )
    .collect()[0]
)

print("Quarantine cause breakdown:")
print(f"  Failed amount: {totals.failed_amount}")  
display(quarantine_with_flags)
